In [2]:
from gradient_clinic.data import prepare_data


In [9]:
from gradient_clinic.models import TinyNet

In [17]:
from gradient_clinic.diagnostics import (
    gradient_norm,
    gradient_stats,
    has_nonfinite_gradients,
    has_nonfinite_parameters
)

In [22]:
from gradient_clinic.evaluation import (
    predict_probabilities,
    classification_metrics,
    evaluate_model
)

In [3]:
data = prepare_data(
    n_samples=7000,
    n_features=20,
    batch_size=128,
    seed=42
)

In [4]:
print(
    "Train:",
    data["X_train"].shape
)

print(
    "Validation:",
    data["X_val"].shape
)

print(
    "Test:",
    data["X_test"].shape
)

Train: (4200, 20)
Validation: (1400, 20)
Test: (1400, 20)


In [5]:
print(
    data["y_train"].shape,
    data["y_val"].shape,
    data["y_test"].shape
)

(4200,) (1400,) (1400,)


In [6]:
xb, yb = next(
    iter(data["train_loader"])
)

print(
    xb.shape,
    yb.shape
)

torch.Size([128, 20]) torch.Size([128])


In [7]:
print(
    data["X_train"].mean(axis=0)[:5]
)

print(
    data["X_train"].std(axis=0)[:5]
)

[-5.0734905e-09 -3.0859596e-08 -9.0826124e-10 -1.6374425e-08
 -9.1393790e-09]
[1.         1.0000004  0.99999964 1.0000001  1.0000006 ]


In [8]:
print(
    data["X_val"].mean(axis=0)[:5]
)

[ 0.02101314 -0.01245438 -0.03536206 -0.00032568 -0.01353869]


In [10]:
model = TinyNet(
    n_features=data["X_train"].shape[1]
)

model

TinyNet(
  (network): Sequential(
    (0): Linear(in_features=20, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.1, inplace=False)
    (7): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [11]:
logits = model(xb)

print(logits.shape)

torch.Size([128])


In [12]:
print(logits[:5])

tensor([-0.2135,  0.0508, -0.1808,  0.0325, -0.0100], grad_fn=<SliceBackward0>)


In [13]:
import torch

probabilities = torch.sigmoid(
    logits
)

print(probabilities[:5])

tensor([0.4468, 0.5127, 0.4549, 0.5081, 0.4975], grad_fn=<SliceBackward0>)


In [14]:
n_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters:",
    n_parameters
)

Trainable parameters: 3585


In [15]:
for name, parameter in model.named_parameters():

    print(
        name,
        parameter.shape
    )

network.0.weight torch.Size([64, 20])
network.0.bias torch.Size([64])
network.1.weight torch.Size([64])
network.1.bias torch.Size([64])
network.4.weight torch.Size([32, 64])
network.4.bias torch.Size([32])
network.7.weight torch.Size([1, 32])
network.7.bias torch.Size([1])


In [18]:
import torch
import torch.nn as nn

loss_fn = nn.BCEWithLogitsLoss()

xb, yb = next(
    iter(data["train_loader"])
)

model.train()

model.zero_grad()

logits = model(xb)

loss = loss_fn(
    logits,
    yb
)

loss.backward()

In [19]:
print(
    "Loss:",
    loss.item()
)

print(
    "Global gradient norm:",
    gradient_norm(model)
)

print(
    "Non-finite gradients:",
    has_nonfinite_gradients(model)
)

print(
    "Non-finite parameters:",
    has_nonfinite_parameters(model)
)

Loss: 0.6811608672142029
Global gradient norm: 0.16336680062589154
Non-finite gradients: False
Non-finite parameters: False


In [20]:
import pandas as pd

grad_df = pd.DataFrame(
    gradient_stats(model)
)

grad_df

,parameter,grad_norm,weight_norm,grad_weight_ratio,gradient_finite
0,network.0.weight,6.215085e-02,4.602606,1.350340e-02,True
1,network.0.bias,2.084135e-09,0.998576,2.087108e-09,True
2,network.1.weight,7.717699e-03,8.000000,9.647124e-04,True
3,network.1.bias,7.316394e-03,0.000000,7.316394e+09,True
4,network.4.weight,1.164539e-01,3.293057,3.536347e-02,True
5,network.4.bias,1.982246e-02,0.374572,5.292026e-02,True
6,network.7.weight,8.047549e-02,0.555041,1.449900e-01,True
7,network.7.bias,4.777181e-02,0.042279,1.129911e+00,True


In [23]:
result = evaluate_model(
    model,
    data["X_val"],
    data["y_val"]
)

In [24]:
result["metrics"]

{'roc_auc': 0.4945588244678475,
 'pr_auc': 0.39205999149760096,
 'brier': 0.2485218197107315}

In [25]:
result["probabilities"][:10]

array([0.45656213, 0.5085147 , 0.48022133, 0.5020982 , 0.520709  ,
       0.47240335, 0.470091  , 0.49715945, 0.51172835, 0.49306607],
      dtype=float32)